# Knowledge Graph Construction
## What is a knowledge graph?
The graph has two interconnected layers:

### 1. Lexical Subgraph (Document Structure)
This models the physical structure of the source papers. It preserves provenance: every extracted fact traces back to a specific text passage.
-  *Document*: One node per paper (so 879 in total). It holds metadata.
 *Chunk*: One node per text chunk (31,704 total). Holds the text content and links to its parent Document.
- *Relationships:*
  - `PART_OF`: Chunk -> Document (which paper does this chunk belong to?)
  - `NEXT_CHUNK`: Chunk -> Chunk (reading order within a paper)

### 2. Domain Subgraph (Scientific Knowledge)
Models the actual biological entities and their relationships, extracted by an LLM from the chunk text.

- *AlgalTaxon*: Species, genera, families (e.g., *Ulva pertusa*, *Porphyra*, Rhodophyta)
- *ChemicalCompound*: Metabolites, pigments, toxins, polysaccharides (e.g., fucoxanthin, carrageenan)
- *BiologicalProcess*: Physiological or ecological events (e.g., photosynthesis, harmful algal bloom)
- *Environment*: Habitats, biomes, geographic locations (e.g., intertidal zone, Yellow Sea)
- *ExperimentalParameter*: Measured/manipulated variables (e.g., salinity at 20 psu, temperature)
- *GeneticSequence*: Molecular markers used in phylogenetics (e.g., rbcL, ITS, SSU)
- *Method*: Techniques and procedures (e.g., ddPCR, HPLC, morphological identification)
- *Application*: Industrial or practical uses (e.g., biofuel production, nutraceuticals)


### 3. The Bridge: MENTIONS
The `MENTIONS` relationship connects the two subgraphs. When a Chunk contains text about *Porphyra birdiae*, I create:
*
(Chunk)-[:MENTIONS]->(AlgalTaxon {scientific_name: "Porphyra birdiae"})
*
This is what makes the graph useful for RAG: it is possible to traverse from a domain concept back to the exact source text that supports it.


| Relation | Description | Example | RO Mapping |
|----------|-------------|---------|------------|
| FOUND_IN | Species located in environment or geographic location | *Ulva pertusa* → intertidal zone | has habitat (RO:0002303) |
| PRODUCES | Species biosynthesizes or yields compound | *P. tricornutum* → fucoxanthin | has output (RO:0002234) |
| STUDIED_WITH | Species analyzed using method or technique | *Porphyra* → rbcL sequencing | Domain-specific |
| IDENTIFIED_BY | Species identified using genetic marker | *P. birdiae* → SSU marker | Domain-specific |
| BELONGS_TO | Taxonomic hierarchy (species → genus → family) | *P. birdiae* → Porphyra | in taxon (RO:0002162) |
| AFFECTS | Parameter or condition influences process or organism | salinity → growth rate | regulates (RO:0002211) |
| CONTAINS | Species contains or is source of compound | Sargassum → alginate | has part (BFO:0000051) |

## Why This Structure?

1. *Traceability*: Every domain node connects back to the chunks that mention it, and those chunks connect to their source documents. The LLM can cite its sources.

2. *Multi-hop reasoning*: Questions like "What methods are used to study HAB-causing species?" require traversing: Method → AlgalTaxon → BiologicalProcess. Vector search can't do this.

3. *Entity unification*: When 50 papers mention *Ulva pertusa*, they all link to one node. The graph aggregates knowledge across the corpus.

## What I am building

1. *Lexical subgraph*: Load all Documents and Chunks from my existing JSONs
2. *LLM extraction*: For each chunk, extract entities and relationships using DeepSeek
3. *Domain subgraph*: Create/merge domain nodes, create MENTIONS edges
4. *Entity resolution*: Deduplicate synonyms (e.g., "U. pertusa" = "Ulva pertusa")
5. *Indexing*: Add constraints and indexes for query performance

## 1 Lexical subgraph

In [41]:
import json
import os
from pathlib import Path

CHUNKS_DIR = Path("../data/chunks/recursive_1000")

# Load all JSON files
json_files = list(CHUNKS_DIR.glob("*.json"))
print(f"Found {len(json_files)} documents")

Found 868 documents


In [2]:
# Load one to inspect structure
with open(json_files[-1]) as f:
    sample_doc = json.load(f)

print(f"\nSample document: {sample_doc['filename']}")
print(f"Title: {sample_doc['title']}")
print(f"Authors: {sample_doc['authors']}")
print(f"Year: {sample_doc['year']}")
print(f"Chunks: {sample_doc['num_chunks']}")


Sample document: algae-2024-39-6-7.pdf
Title: Effects of aeration and centrifugation conditions on omega-3 fatty acid production by the mixotrophic dinoflagellate Gymnodinium smaydae in a semi-continuous cultivation system on a pilot scale
Authors: ['Ji Hyun You1, Hae Jin Jeong1,*, Sang Ah Park1, Se Hee Eom1, Hee Chang Kang1 and Jin Hee Ok3']
Year: 2024
Chunks: 64


In [3]:
# for node metadata
def filename_to_doi(filename):
    # algae-2002-17-4-203.pdf -> 10.4490/algae.2002.17.4.203
    stem = filename.replace(".pdf", "").replace(".json", "")
    parts = stem.split("-")  # ["algae", "2002", "17", "4", "203"]
    return f"10.4490/{parts[0]}.{parts[1]}.{parts[2]}.{parts[3]}.{parts[4]}"

## 2. LLM Extraction: Extract Entities and Relationships

### 2.1 Test on a single chunk

Verify the extraction prompt works before running at scale.

In [ ]:
# Cell 1: Imports and config
import json
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

In [19]:
# Cell 2: Load sample document
json_files = sorted(CHUNKS_DIR.glob("*.json"), reverse=True)  # newest first so I dont have to write -1
print(f"Found {len(json_files)} documents")

with open(json_files[0]) as f:
    sample_doc = json.load(f)

print(f"Sample: {sample_doc['filename']}")
print(f"Chunks: {sample_doc['num_chunks']}")

Found 868 documents
Sample: algae-2024-39-6-7.pdf
Chunks: 64


In [20]:
# Cell 3: Preview a chunk
test_chunk = sample_doc["chunks"][1]["text"]
print(test_chunk[:500])

High production and efficient harvesting of microalgae containing high omega-3 levels are critical concerns for indus- trial use. Aeration can elevate production of some microalgae by providing CO2 and O2. However, it may lower the pro- duction of others by generating shear stress, causing severe cell damage. The mixotrophic dinoflagellate Gymnodinium smaydae is a new, promising microalga for omega-3 fatty acid production owing to its high docosahexaenoic acid con- tent, and determining optimal 


In [21]:
# Cell 4: Set up API client with instructor
import instructor
from openai import OpenAI

client = instructor.from_openai(
    OpenAI(
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url="https://api.deepseek.com"
    ),
    mode=instructor.Mode.JSON
)

In [22]:
# Cell 5: Define entity and relation types
from typing import Literal, Optional
from pydantic import BaseModel, Field
#Old Solution:
"""ENTITY_TYPES = ["AlgalTaxon", "ChemicalCompound", "Method", "Environment", 
                "GeneticMarker", "Application", "BiologicalProcess", "ExperimentalParameter"]

RELATION_TYPES = ["FOUND_IN", "PRODUCES", "STUDIED_WITH", "IDENTIFIED_BY", 
                  "BELONGS_TO", "AFFECTS", "CONTAINS"]
                  """
# Defines Pydantic schemas (Improved new solution based on notebook lm research)
# --------------------------------------------------------------------------
# Entity types with DISTINCT field names to prevent LLM pattern confusion
# --------------------------------------------------------------------------

class AlgalTaxon(BaseModel):
    """A species, genus, or taxonomic group of algae."""
    species_name: str = Field(description="Scientific name, e.g., 'Ulva pertusa'")
    taxonomic_rank: Optional[str] = Field(default=None, description="e.g., species, genus, family")

class ChemicalCompound(BaseModel):
    """A metabolite, pigment, toxin, or polysaccharide."""
    compound_name: str = Field(description="Chemical name, e.g., 'fucoxanthin'")

class Method(BaseModel):
    """An experimental technique or analytical procedure."""
    method_name: str = Field(description="Technique name, e.g., 'HPLC', 'rbcL sequencing'")

class Environment(BaseModel):
    """A habitat, biome, or geographic location."""
    environment_name: str = Field(description="Location or habitat, e.g., 'Yellow Sea', 'intertidal zone'")

class GeneticMarker(BaseModel):
    """A molecular marker used for identification or phylogenetics."""
    marker_name: str = Field(description="Marker name, e.g., 'rbcL', 'ITS', 'SSU'")

class Application(BaseModel):
    """An industrial or practical use of algae."""
    application_name: str = Field(description="Use case, e.g., 'biofuel production'")

# --------------------------------------------------------------------------
# Relationship with confidence score for quality filtering
# --------------------------------------------------------------------------

RelationType = Literal[
    "FOUND_IN", "PRODUCES", "STUDIED_WITH", 
    "IDENTIFIED_BY", "BELONGS_TO", "AFFECTS", "CONTAINS"
]

class Relationship(BaseModel):
    """A relationship between two entities, with confidence score."""
    subject: str = Field(description="Source entity name (exact as extracted)")
    predicate: RelationType = Field(description="Relationship type")
    object: str = Field(description="Target entity name (exact as extracted)")
    confidence: float = Field(
        description="Confidence score 0.0-1.0 based on how explicitly stated in text",
        ge=0.0, le=1.0
    )

# --------------------------------------------------------------------------
# Nested response model - the "tackle box" that organizes LLM attention
# --------------------------------------------------------------------------

class ExtractionResult(BaseModel):
    """
    Complete extraction output for one chunk.
    
    Nested structure reduces LLM cognitive load by providing
    distinct 'drawers' for each entity type.
    """
    taxa: list[AlgalTaxon] = Field(default_factory=list)
    compounds: list[ChemicalCompound] = Field(default_factory=list)
    methods: list[Method] = Field(default_factory=list)
    environments: list[Environment] = Field(default_factory=list)
    markers: list[GeneticMarker] = Field(default_factory=list)
    applications: list[Application] = Field(default_factory=list)
    relationships: list[Relationship] = Field(default_factory=list)

In [23]:
# Cell: Extraction function with updated prompt

SYSTEM_PROMPT = """You are a precise scientific information extractor for phycology (algae research).

For each entity you extract, place it in the correct category.
For each relationship, include a confidence score (0.0-1.0) based on how explicitly it is stated.
- 1.0 = directly stated ("Ulva pertusa was found in the Yellow Sea")
- 0.5 = implied but not explicit
- Below 0.3 = weak inference

Only extract what is explicitly stated. Return empty lists if nothing relevant found."""

def extract_from_chunk(chunk_text: str) -> ExtractionResult:
    """Extract entities and relationships with confidence scores."""
    return client.chat.completions.create(
        model="deepseek-chat",
        response_model=ExtractionResult,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Extract entities and relationships from:\n\n{chunk_text}"}
        ],
        temperature=0.1 # for deterministic outputs
    )

### 2.2 Test on a full paper

Check consistency across all chunks in one document.

In [24]:
# Cell: Pilot batch validation (Section 2.2)
# Runs on ~100 chunks, check for orphans and confidence distribution
import time
def run_pilot_batch(json_files: list, n_chunks: int = 100) -> list[dict]:
    """
    Runs extraction on a small pilot batch.
    Returns list of results for validation.
    """
    results = []
    chunk_count = 0
    
    for json_file in json_files:
        if chunk_count >= n_chunks:
            break
            
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        
        for chunk in doc["chunks"]:
            if chunk_count >= n_chunks:
                break
            
            chunk_id = f"{doc['filename'].replace('.pdf', '')}_chunk_{chunk['chunk_id']:03d}"
            
            try:
                result = extract_from_chunk(chunk["text"])
                time.sleep(0.5)  # Here to prevent api rate limits
                results.append({
                    "chunk_id": chunk_id,
                    "extraction": result.model_dump(),
                    "error": None
                })
            except Exception as e:
                results.append({
                    "chunk_id": chunk_id,
                    "extraction": None,
                    "error": str(e)
                })
            
            chunk_count += 1
            if chunk_count % 10 == 0:
                print(f"Processed {chunk_count}/{n_chunks} chunks...")
    
    return results

In [25]:
# Cell: Validation metrics

def validate_pilot(results: list[dict]) -> dict:
    """
    Check pilot batch for quality issues.
    Returns metrics dict with warnings.
    """
    total = len(results)
    errors = sum(1 for r in results if r["error"])
    
    # Count entities and relationships
    total_entities = 0
    total_relationships = 0
    low_confidence_rels = 0
    orphan_entities = 0  # entities with no relationships
    
    for r in results:
        if r["extraction"] is None:
            continue
        
        ext = r["extraction"]
        
        # Count all entities across categories
        entity_names = set()
        for category in ["taxa", "compounds", "methods", "environments", "markers", "applications"]:
            name_field = {
            "taxa": "species_name",
            "compounds": "compound_name", 
            "methods": "method_name",
            "environments": "environment_name",
            "markers": "marker_name",
            "applications": "application_name"}[category]

            for entity in ext.get(category, []):
                # Get the name field (each has distinct naming)
                name = entity.get(name_field)  
                entity_names.add(name)
                total_entities += 1
        
        # Count relationships and check confidence
        rels = ext.get("relationships", [])
        total_relationships += len(rels)
        
        for rel in rels:
            if rel.get("confidence", 1.0) < 0.5:
                low_confidence_rels += 1
        
        # Check for orphan entities (not in any relationship)
        rel_entities = set()
        for rel in rels:
            rel_entities.add(rel.get("subject", ""))
            rel_entities.add(rel.get("object", ""))# previously rel_entities.add(rel["object"])
        
        orphan_entities += len(entity_names - rel_entities)
    
    return {
        "total_chunks": total,
        "errors": errors,
        "error_rate": errors / total if total > 0 else 0,
        "total_entities": total_entities,
        "total_relationships": total_relationships,
        "entity_to_rel_ratio": total_entities / total_relationships if total_relationships > 0 else float('inf'),
        "low_confidence_relationships": low_confidence_rels,
        "orphan_entities": orphan_entities,
        "orphan_rate": orphan_entities / total_entities if total_entities > 0 else 0
    }

In [26]:
# Full paper test
#  Run pilot batch and validate

# Load files sorted newest first (better OCR quality)
json_files = sorted(CHUNKS_DIR.glob("*.json"), reverse=True)

# Run on 100 chunks
pilot_results = run_pilot_batch(json_files, n_chunks=100)

# Check quality metrics
metrics = validate_pilot(pilot_results)

print("*Pilot Batch Results*")
print(f"Chunks processed: {metrics['total_chunks']}")
print(f"Errors: {metrics['errors']} ({metrics['error_rate']:.1%})")
print(f"Total entities: {metrics['total_entities']}")
print(f"Total relationships: {metrics['total_relationships']}")
print(f"Entity-to-relationship ratio: {metrics['entity_to_rel_ratio']:.2f}")
print(f"Low confidence relationships (<0.5): {metrics['low_confidence_relationships']}")
print(f"Orphan entities: {metrics['orphan_entities']} ({metrics['orphan_rate']:.1%})")

# Warnings
if metrics['error_rate'] > 0.1:
    print("\n!!!  High error rate - check API connection or prompt")
if metrics['entity_to_rel_ratio'] < 1.0:
    print("\n!!!  Low entity-to-relationship ratio - might indicate extraction issues")
if metrics['orphan_rate'] > 0.5:
    print("\n!!!  High orphan rate - relationships not being extracted properly")

Processed 10/100 chunks...
Processed 20/100 chunks...


KeyboardInterrupt: 

In [ ]:
# Checks what's actually in the extractions
import pprint
sample = pilot_results[0]["extraction"]
print("Keys in extraction:", sample.keys())
print("\nFull structure:")

pprint.pprint(sample)

Keys in extraction: dict_keys(['taxa', 'compounds', 'methods', 'environments', 'markers', 'applications', 'relationships'])

Full structure:
{'applications': [{'application_name': 'omega-3 fatty acid production'}],
 'compounds': [{'compound_name': 'omega-3 fatty acid'}],
 'environments': [],
 'markers': [],
 'methods': [{'method_name': 'aeration'},
             {'method_name': 'centrifugation'},
             {'method_name': 'semi-continuous cultivation'},
             {'method_name': 'pilot scale cultivation'}],
 'relationships': [{'confidence': 1.0,
                    'object': 'omega-3 fatty acid',
                    'predicate': 'PRODUCES',
                    'subject': 'Gymnodinium smaydae'},
                   {'confidence': 1.0,
                    'object': 'omega-3 fatty acid production',
                    'predicate': 'AFFECTS',
                    'subject': 'aeration'},
                   {'confidence': 1.0,
                    'object': 'omega-3 fatty acid production',

### 2.3 Run full extraction with caching

The extraction pipeline is built around two functions that separate concerns: one handles a single chunk with caching, the other orchestrates the full corpus iteration.

### How `extract_with_cache` works

This function processes one chunk and ensures we never pay for the same extraction twice:

1. **Build the cache path** from the chunk ID (e.g., `data/kg_extractions/algae-2024-39-1-15_chunk_007.json`)
2. **Check if cached result exists**: if yes, load and return it without calling the API
3. **If not cached**, call `extract_from_chunk` which sends the text to DeepSeek via the instructor-wrapped client
4. **Convert the Pydantic result to a dict** using `.model_dump()` so it's JSON-serializable
5. **Save to cache directory** so subsequent runs skip this chunk
6. **Return the extraction dict** for immediate use or aggregation

The caching is file-based rather than in-memory, which means the extraction can be interrupted at any point (laptop dies, API rate limit, budget pause) and resumed later without re-processing or re-paying for completed chunks.


In [27]:
def extract_with_cache(chunk_id, chunk_text, cache_dir="data/kg_extractions"):
    cache_path = os.path.join(cache_dir, f"{chunk_id}.json")
    
    # Already extracted? Return cached result
    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)
    
    # Call API
    result = extract_from_chunk(chunk_text)
    
    # Cache it
    os.makedirs(cache_dir, exist_ok=True)
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(result.model_dump(), f)
    
    return result.model_dump()

### How `run_full_extraction` works

This function iterates through the corpus with progress tracking and error resilience:

1. **Accept sorted JSON files** — we pass these newest-first to prioritize recent papers with better OCR quality
2. **Iterate through documents and their chunks** — nested loop since each JSON contains one paper's worth of chunks
3. **Build a unique chunk ID** from filename and chunk index (e.g., `algae-2024-39-1-15_chunk_007`)
4. **Check cache before calling** — if the file exists, count as skipped; otherwise process and count as new
5. **Rate limit only on actual API calls** — `time.sleep(0.5)` runs only when we hit DeepSeek, not on cache hits
6. **Catch exceptions per-chunk** — one bad chunk doesn't kill the whole run; errors are logged and counted
7. **Progress reporting every 50 chunks** — so you can monitor cost accumulation and estimate completion time
8. **Return statistics** — processed, skipped, and error counts for validation

In [28]:
# Full loop with caching

def run_full_extraction(json_files: list, max_chunks: int = None, cache_dir: str = "data/kg_extractions"):
    """
    Extract entities and relationships from all chunks with caching.
    Can be interrupted and resumed - already processed chunks are skipped.
    """
    processed = 0
    skipped = 0
    errors = 0
    chunk_count = 0
    
    for json_file in json_files:
        if max_chunks and chunk_count >= max_chunks:
            break
        
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        
        doc_id = doc["filename"].replace(".pdf", "")
        
        for chunk in doc["chunks"]:
            if max_chunks and chunk_count >= max_chunks:
                break
            
            chunk_id = f"{doc_id}_chunk_{chunk['chunk_id']:03d}"
            cache_path = os.path.join(cache_dir, f"{chunk_id}.json")
            
            # Check if already cached (for skip counting)
            already_cached = os.path.exists(cache_path)
            
            try:
                extract_with_cache(chunk_id, chunk["text"], cache_dir)
                
                if already_cached:
                    skipped += 1
                else:
                    processed += 1
                    time.sleep(0.5)  # Only rate limit on actual API calls
                    
            except Exception as e:
                print(f"Error on {chunk_id}: {e}")
                errors += 1
            
            chunk_count += 1
            
            if processed % 50 == 0 and processed > 0:
                print(f"Processed: {processed} | Skipped: {skipped} | Errors: {errors}")
    
    print(f"\n~Extraction Complete~")
    print(f"Processed: {processed}")
    print(f"Skipped (cached): {skipped}")
    print(f"Errors: {errors}")
    
    return {"processed": processed, "skipped": skipped, "errors": errors}

### Why this design

The file-per-chunk caching strategy has several advantages for a project with budget constraints:

- **Resumability**: Stops anytime, restarts without cost penalty
- **Inspectability**: I can manually examine any extraction by opening its JSON file
- **Parallelizability**: If I later get access to UCloud or multiple machines, different processes can work on different chunks without coordination (file existence is the lock)
- **Debuggability**: If something looks wrong in Neo4j, it can eb traced back to the exact cached extraction that produced it

In [29]:
#call
stats = run_full_extraction(
    json_files, 
    max_chunks=3000,
    cache_dir="data/kg_extractions"
)

KeyboardInterrupt: 

## 3. Domain Subgraph: Load into Neo4j

This section transforms the cached LLM extractions into a queryable knowledge graph. The ingestion happens in two phases that mirror the bipartite architecture described at the start of this notebook.

### How `create_lexical_subgraph` works

This function builds the document structure layer that preserves provenance:

1. **Iterate through source JSONs** — each file represents one paper with its chunked text
2. **Filter to extracted documents** — only create nodes for papers where at least one chunk was processed by the LLM (no point creating empty document shells)
3. **Create Document nodes** — using MERGE to avoid duplicates, storing metadata (DOI, title, authors, year)
4. **Create Chunk nodes with PART_OF edges** — each chunk links to its parent document, enabling "which paper said this?" queries
5. **Create NEXT_CHUNK edges** — sequential links between chunks preserve reading order, allowing context expansion during retrieval

The MERGE command is critical here: if the same document or chunk is encountered again (e.g., during a resumed run), Neo4j binds to the existing node rather than creating a duplicate.

### How `create_domain_subgraph` works

This function builds the scientific knowledge layer from the LLM extractions:

1. **Map categories to labels** — the `category_config` dict translates Pydantic field names (`taxa`, `compounds`) to Neo4j labels (`AlgalTaxon`, `ChemicalCompound`) and their corresponding name fields
2. **Create entity nodes** — for each extracted entity, MERGE a node with its name. MERGE ensures that when 50 papers mention "Ulva pertusa", they all connect to one node rather than 50 duplicates
3. **Create MENTIONS edges** — the bridge between subgraphs. Each chunk links to every entity it contains, enabling "show me the source text for this claim" traversals
4. **Create domain relationships** — the extracted relationships (PRODUCES, FOUND_IN, etc.) become edges between entity nodes, with confidence scores stored as edge properties
5. **Filter low-confidence relationships** — anything below 0.5 confidence is skipped to keep the graph clean

### Why this two-phase approach

Separating lexical and domain ingestion has practical benefits:

- **Debuggability**: If entity extraction looks wrong, you can wipe domain nodes without losing the document structure
- **Incremental updates**: New extractions can be added without rebuilding the entire graph
- **Clear provenance**: The MENTIONS edges form an explicit audit trail from any scientific claim back to its source text

In [31]:
#Neo4j connection
from neo4j import GraphDatabase

NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "graphrag"  # replace with yours

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# Test connection
with driver.session() as session:
    result = session.run("RETURN 1 AS test")
    print("Connected to Neo4j:", result.single()["test"] == 1)

Connected to Neo4j: True


In [32]:
# Cell: Cache directory for extractions
CACHE_DIR = Path("data/kg_extractions")

In [48]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
saved = 0
for result in pilot_results:
    if result["extraction"] is not None:
        cache_path = CACHE_DIR / f"{result['chunk_id']}.json"
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(result["extraction"], f)
        saved += 1

print(f"Saved {saved} extractions to cache")

Saved 99 extractions to cache


In [49]:
# Cell: Load cached extractions

cached_files = list(CACHE_DIR.glob("*.json"))
print(f"Found {len(cached_files)} cached extractions")

extractions = {}
for cache_file in cached_files:
    chunk_id = cache_file.stem
    with open(cache_file, encoding="utf-8") as f:
        extractions[chunk_id] = json.load(f)

print(f"Loaded {len(extractions)} extractions")

Found 99 cached extractions
Loaded 99 extractions


In [50]:
# Cell: Create lexical subgraph (Documents and Chunks)

def create_lexical_subgraph(session, json_files, extractions):
    """
    Create Document and Chunk nodes, plus PART_OF and NEXT_CHUNK relationships.
    Only creates nodes for documents that have cached extractions.
    """
    docs_created = 0
    chunks_created = 0
    
    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)
        
        doc_id = doc["filename"].replace(".pdf", "")
        doi = filename_to_doi(doc["filename"])
        
        # Check if any chunks from this doc were extracted
        doc_chunk_ids = [f"{doc_id}_chunk_{c['chunk_id']:03d}" for c in doc["chunks"]]
        extracted_chunk_ids = [cid for cid in doc_chunk_ids if cid in extractions]
        
        if not extracted_chunk_ids:
            continue
        
        # Create Document node
        session.run("""
            MERGE (d:Document {doi: $doi})
            SET d.title = $title,
                d.authors = $authors,
                d.year = $year,
                d.filename = $filename
        """, doi=doi, title=doc["title"], authors=doc["authors"], 
            year=doc["year"], filename=doc["filename"])
        docs_created += 1
        
        # Create Chunk nodes and relationships
        prev_chunk_id = None
        for chunk in doc["chunks"]:
            chunk_id = f"{doc_id}_chunk_{chunk['chunk_id']:03d}"
            
            if chunk_id not in extractions:
                continue
            
            # Create Chunk node with PART_OF relationship to Document
            session.run("""
                MERGE (c:Chunk {chunk_id: $chunk_id})
                SET c.text = $text
                WITH c
                MATCH (d:Document {doi: $doi})
                MERGE (c)-[:PART_OF]->(d)
            """, chunk_id=chunk_id, text=chunk["text"], doi=doi)
            chunks_created += 1
            
            # Create NEXT_CHUNK relationship
            if prev_chunk_id and prev_chunk_id in extractions:
                session.run("""
                    MATCH (c1:Chunk {chunk_id: $prev_id})
                    MATCH (c2:Chunk {chunk_id: $curr_id})
                    MERGE (c1)-[:NEXT_CHUNK]->(c2)
                """, prev_id=prev_chunk_id, curr_id=chunk_id)
            
            prev_chunk_id = chunk_id
        
        if docs_created % 20 == 0:
            print(f"Documents: {docs_created}, Chunks: {chunks_created}")
    
    return docs_created, chunks_created

In [51]:
# Cell: Create domain subgraph (entities and relationships)

def create_domain_subgraph(session, extractions):
    """
    Create domain entity nodes, MENTIONS edges, and domain relationships.
    """
    entities_created = 0
    mentions_created = 0
    rels_created = 0
    
    # Maps category to label and name field
    category_config = {
        "taxa": ("AlgalTaxon", "species_name"),
        "compounds": ("ChemicalCompound", "compound_name"),
        "methods": ("Method", "method_name"),
        "environments": ("Environment", "environment_name"),
        "markers": ("GeneticMarker", "marker_name"),
        "applications": ("Application", "application_name")
    }
    
    for chunk_id, extraction in extractions.items():
        # Track entity names for MENTIONS edges
        chunk_entity_names = []
        
        # Create entity nodes
        for category, (label, name_field) in category_config.items():
            for entity in extraction.get(category, []):
                name = entity.get(name_field)
                if not name:
                    continue
                
                chunk_entity_names.append((label, name))
                
                # MERGE entity node
                session.run(f"""
                    MERGE (e:{label} {{name: $name}})
                """, name=name)
                entities_created += 1
        
        # Create MENTIONS edges
        for label, name in chunk_entity_names:
            session.run(f"""
                MATCH (c:Chunk {{chunk_id: $chunk_id}})
                MATCH (e:{label} {{name: $name}})
                MERGE (c)-[:MENTIONS]->(e)
            """, chunk_id=chunk_id, name=name)
            mentions_created += 1
        
        # Create domain relationships
        for rel in extraction.get("relationships", []):
            if rel.get("confidence", 0) < 0.5:
                continue
            
            session.run("""
                MATCH (s {name: $subject})
                MATCH (o {name: $object})
                MERGE (s)-[r:""" + rel["predicate"] + """]->(o)
                SET r.confidence = $confidence
            """, subject=rel["subject"], object=rel["object"], 
                confidence=rel["confidence"])
            rels_created += 1
        
        if len(extractions) > 50 and entities_created % 500 == 0:
            print(f"Entities: {entities_created}, Mentions: {mentions_created}, Rels: {rels_created}")
    
    return entities_created, mentions_created, rels_created

In [52]:
# Cell: Run ingestion

with driver.session() as session:
    print("Creating lexical subgraph...")
    docs, chunks = create_lexical_subgraph(session, json_files, extractions)
    print(f"Created {docs} documents, {chunks} chunks")
    
    print("\nCreating domain subgraph...")
    entities, mentions, rels = create_domain_subgraph(session, extractions)
    print(f"Created {entities} entities, {mentions} mentions, {rels} relationships")

print("\nDone!")

Creating lexical subgraph...
Created 2 documents, 99 chunks

Creating domain subgraph...
Created 551 entities, 551 mentions, 436 relationships

Done!


In [44]:
# Debug: find the problematic filename
for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        doc = json.load(f)
    parts = doc["filename"].replace(".pdf", "").split("-")
    if len(parts) < 5:
        print(f"Bad filename: {doc['filename']} -> {parts}")

In [45]:
for json_file in json_files:
    with open(json_file, encoding="utf-8") as f:
        doc = json.load(f)
    parts = doc["filename"].replace(".pdf", "").split("-")
    if len(parts) < 5:
        print(f"File on disk: {json_file.name}")
        print(f"Internal filename: {doc['filename']}")
        print()